# 🧠 StudyAssist — Remote Colab Brain Server

> ⚠️ **Important Google Colab Terms of Service (ToS) Notice**:
> Running tunnels or remote proxy servers on Google Colab free tier may be subject to session limits, inactivity disconnects, and terms of service policies regarding remote service hosting. Use this notebook responsibly for your own personal educational study sessions. Do not leave unattended for extended periods.

This notebook runs **Ollama with Qwen 2.5 7B** on a free Colab GPU (T4 / V100 / A100) and exposes an authenticated tunnel back to your local **StudyAssist** desktop application.

## Step 1: Install Ollama and Background Service

In [ ]:
# 1. Install Ollama
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start Ollama daemon in background
import subprocess
import time

ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
print("✅ Ollama daemon launched!")

## Step 2: Pull Study Model (Qwen 2.5 7B)

In [ ]:
MODEL_NAME = "qwen2.5:7b"
print(f"Pulling model '{MODEL_NAME}' onto Colab GPU...")
!ollama pull {MODEL_NAME}
print(f"✅ Model {MODEL_NAME} is ready in Colab memory!")

## Step 3: Launch Bearer-Token Authenticated Reverse Proxy

In [ ]:
import http.server
import json
import secrets
import threading
import urllib.request

# Generate strong one-time bearer token
AUTH_TOKEN = secrets.token_hex(16)
PROXY_PORT = 8000

class AuthProxyHandler(http.server.BaseHTTPRequestHandler):
    def do_POST(self):
        auth_header = self.headers.get("Authorization", "")
        expected = f"Bearer {AUTH_TOKEN}"
        if auth_header != expected:
            self.send_response(401)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(b'{"error": "Unauthorized: Invalid bearer token"}')
            return

        length = int(self.headers.get("Content-Length", 0))
        req_body = self.rfile.read(length)
        target_url = f"http://localhost:11434{self.path}"

        req = urllib.request.Request(
            target_url,
            data=req_body,
            headers={"Content-Type": "application/json"},
            method="POST"
        )
        try:
            with urllib.request.urlopen(req, timeout=120) as resp:
                resp_data = resp.read()
                self.send_response(resp.status)
                self.send_header("Content-Type", "application/json")
                self.end_headers()
                self.wfile.write(resp_data)
        except Exception as e:
            self.send_response(500)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(json.dumps({"error": str(e)}).encode("utf-8"))

    def do_GET(self):
        # Healthcheck endpoint
        if self.path in ["/v1/models", "/healthz", "/"]:
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(b'{"status": "ok", "service": "StudyAssist Colab Tunnel"}')
        else:
            self.send_response(404)
            self.end_headers()

server = http.server.ThreadingHTTPServer(("0.0.0.0", PROXY_PORT), AuthProxyHandler)
proxy_thread = threading.Thread(target=server.serve_forever, daemon=True)
proxy_thread.start()
print(f"✅ Authenticated proxy server running on port {PROXY_PORT}!")

## Step 4: Open Public Tunnel via Cloudflare

In [ ]:
# Download cloudflared binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

# Start cloudflared quick tunnel in background
import re
import subprocess

cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PROXY_PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

tunnel_url = None
for _ in range(30):
    line = cf_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(1)

print("\n" + "="*65)
print("🚀 STUDYASSIST COLAB TUNNEL IS READY!")
print("="*65)
print(f"Base URL : {tunnel_url}")
print(f"Token    : {AUTH_TOKEN}")
print("="*65)
print("\n👉 Instructions:")
print("1. In your local StudyAssist web UI, navigate to the 'Brain & Settings' tab.")
print("2. Select profile 'colab_ollama'.")
print(f"3. Paste Base URL: {tunnel_url}")
print(f"4. Paste Colab Token: {AUTH_TOKEN}")
print("5. Click 'Save Profile Settings' and 'Test Brain Connection'!")